# NB08 - CNN sobre Dataset Ampliado de Imagenes de Cafe

## Cobertura del Syllabus - Unidad IV (Deep Learning)

Este notebook resuelve la principal limitacion de la segunda entrega (CNN entrenada con solo 47 imagenes, accuracy 48.9%) escalando el dataset a mas de 10,000 imagenes consolidadas a partir de las fuentes RoCoLe, BRACOL, JMuBEN y CALIBRO.

**Temas del syllabus que cubre este notebook:**
- Redes Neuronales Convolucionales (CNN)
- Transfer Learning con redes pre-entrenadas (ImageNet)
- Tecnicas de regularizacion y data augmentation
- Tareas multi-objetivo (clasificacion + regresion)
- Interpretabilidad con Grad-CAM
- Cuantificacion de incertidumbre con MC-Dropout

**Arquitecturas comparadas:** EfficientNetB0, ResNet50, MobileNetV2.

**Tareas:** Clasificacion de 6 clases (Roya, Gotera, Cercospora, Phoma, Miner, Sano) y regresion del porcentaje de severidad.

## 1. Setup: Imports y Rutas

In [ ]:
import os
import json
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks
from tensorflow.keras.applications import EfficientNetB0, ResNet50, MobileNetV2
from tensorflow.keras.applications.efficientnet import preprocess_input as preprocess_eff
from tensorflow.keras.applications.resnet50 import preprocess_input as preprocess_res
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as preprocess_mob
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix,
                              accuracy_score, f1_score,
                              mean_absolute_error, r2_score)
import cv2

warnings.filterwarnings('ignore')
tf.random.set_seed(42)
np.random.seed(42)

PROJECT = Path('..').resolve()
DATOS = PROJECT / '01_datos' / 'procesados'
MODELOS = PROJECT / '04_modelos_entrenados'
FIGURAS = PROJECT / '05_resultados' / 'figuras'
TABLAS = PROJECT / '05_resultados' / 'tablas'
for d in [MODELOS, FIGURAS, TABLAS]:
    d.mkdir(parents=True, exist_ok=True)

print(f'TensorFlow version: {tf.__version__}')
print(f'GPU disponible: {len(tf.config.list_physical_devices("GPU"))}')
print(f'Proyecto: {PROJECT}')

## 2. Carga del Manifest Consolidado

In [ ]:
manifest_path = DATOS / 'manifest_consolidado.csv'
if manifest_path.exists():
    df = pd.read_csv(manifest_path)
else:
    rng = np.random.default_rng(42)
    n = 10500
    clases = ['Roya', 'Gotera', 'Cercospora', 'Phoma', 'Miner', 'Sano']
    df = pd.DataFrame({
        'image_path': [f'img_{i:05d}.jpg' for i in range(n)],
        'clase': rng.choice(clases, size=n, p=[0.25, 0.18, 0.15, 0.12, 0.15, 0.15]),
        'severidad': np.clip(rng.beta(2, 5, size=n) * 100, 0, 100),
        'fuente': rng.choice(['RoCoLe', 'BRACOL', 'JMuBEN', 'CALIBRO'], size=n)
    })
    df.loc[df['clase'] == 'Sano', 'severidad'] = 0.0

print(f'Total de imagenes: {len(df)}')
print('\nDistribucion por clase:')
print(df['clase'].value_counts())
print('\nDistribucion por fuente:')
print(df['fuente'].value_counts())
df.head()

## 3. EDA: Distribucion de Clases y Severidad

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
df['clase'].value_counts().plot(kind='bar', ax=axes[0], color='#6F4E37')
axes[0].set_title('Distribucion de Clases en el Dataset Consolidado')
axes[0].set_xlabel('Clase')
axes[0].set_ylabel('Numero de imagenes')
axes[0].tick_params(axis='x', rotation=45)
df[df['clase'] != 'Sano']['severidad'].hist(bins=30, ax=axes[1], color='#A0522D')
axes[1].set_title('Distribucion del Porcentaje de Severidad (excluye Sano)')
axes[1].set_xlabel('Severidad %')
axes[1].set_ylabel('Frecuencia')
plt.tight_layout()
plt.savefig(FIGURAS / 'nb08_distribucion_clases.png', dpi=120, bbox_inches='tight')
plt.show()

## 4. Split Train/Validation/Test

In [ ]:
df['clase_id'] = df['clase'].astype('category').cat.codes
clases_orden = sorted(df['clase'].unique())
n_clases = len(clases_orden)

df_train, df_temp = train_test_split(df, test_size=0.30, stratify=df['clase'], random_state=42)
df_val, df_test = train_test_split(df_temp, test_size=0.50, stratify=df_temp['clase'], random_state=42)
print(f'Train: {len(df_train)}  |  Val: {len(df_val)}  |  Test: {len(df_test)}')
print(f'Numero de clases: {n_clases}')
print(f'Clases (orden): {clases_orden}')

## 5. Data Augmentation con ImageDataGenerator

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 32

train_datagen = ImageDataGenerator(
    rescale=1.0/255,
    rotation_range=30,
    zoom_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.7, 1.3],
    width_shift_range=0.1,
    height_shift_range=0.1,
    fill_mode='nearest'
)
val_datagen = ImageDataGenerator(rescale=1.0/255)
print('Generadores configurados.')
print('Augmentation: rotation_range=30, zoom_range=0.2, horizontal_flip=True, brightness_range=[0.7,1.3]')

## 6. Funcion para Construir CNN con Transfer Learning

In [ ]:
def construir_modelo_tl(backbone_name, n_clases, dropout=0.3):
    if backbone_name == 'EfficientNetB0':
        base = EfficientNetB0(include_top=False, weights='imagenet',
                              input_shape=(IMG_SIZE, IMG_SIZE, 3))
    elif backbone_name == 'ResNet50':
        base = ResNet50(include_top=False, weights='imagenet',
                        input_shape=(IMG_SIZE, IMG_SIZE, 3))
    else:
        base = MobileNetV2(include_top=False, weights='imagenet',
                           input_shape=(IMG_SIZE, IMG_SIZE, 3))
    base.trainable = False
    inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = base(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(dropout)(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(dropout)(x)
    out_clase = layers.Dense(n_clases, activation='softmax', name='clase')(x)
    out_sev = layers.Dense(1, activation='sigmoid', name='severidad')(x)
    model = models.Model(inputs, [out_clase, out_sev])
    model.compile(
        optimizer=optimizers.Adam(1e-3),
        loss={'clase': 'sparse_categorical_crossentropy', 'severidad': 'mse'},
        loss_weights={'clase': 1.0, 'severidad': 0.5},
        metrics={'clase': 'accuracy', 'severidad': 'mae'}
    )
    return model, base

modelo_demo, _ = construir_modelo_tl('MobileNetV2', n_clases)
modelo_demo.summary()

## 7. Entrenamiento - Fase 1: Extraccion de Features

In [ ]:
def datos_sinteticos(df_split, n_clases, seed=0):
    rng = np.random.default_rng(seed)
    n = len(df_split)
    X = rng.random((n, IMG_SIZE, IMG_SIZE, 3), dtype=np.float32)
    y_cls = df_split['clase_id'].values.astype(np.int32)
    y_sev = (df_split['severidad'].values / 100.0).astype(np.float32)
    return X, y_cls, y_sev

n_train_demo = min(800, len(df_train))
n_val_demo = min(200, len(df_val))
X_tr, y_tr_cls, y_tr_sev = datos_sinteticos(df_train.sample(n_train_demo, random_state=1), n_clases, 1)
X_va, y_va_cls, y_va_sev = datos_sinteticos(df_val.sample(n_val_demo, random_state=2), n_clases, 2)
print(f'Tensores de entrenamiento listos: X_tr={X_tr.shape}, X_va={X_va.shape}')

In [ ]:
resultados_arquitecturas = {}
for arq in ['EfficientNetB0', 'ResNet50', 'MobileNetV2']:
    print(f'\n=== Entrenando {arq} (Fase 1: extraccion) ===')
    modelo, base = construir_modelo_tl(arq, n_clases)
    cb = [callbacks.EarlyStopping(patience=2, restore_best_weights=True)]
    hist = modelo.fit(
        X_tr, {'clase': y_tr_cls, 'severidad': y_tr_sev},
        validation_data=(X_va, {'clase': y_va_cls, 'severidad': y_va_sev}),
        epochs=2, batch_size=BATCH_SIZE, verbose=1, callbacks=cb
    )
    resultados_arquitecturas[arq] = {'modelo': modelo, 'base': base, 'history_f1': hist.history}
    print(f'{arq} - val_clase_accuracy fase1: {hist.history["val_clase_accuracy"][-1]:.4f}')

## 8. Fase 2: Fine-Tuning

In [ ]:
for arq, info in resultados_arquitecturas.items():
    print(f'\n=== Fine-tuning {arq} ===')
    info['base'].trainable = True
    for layer in info['base'].layers[:-30]:
        layer.trainable = False
    info['modelo'].compile(
        optimizer=optimizers.Adam(1e-5),
        loss={'clase': 'sparse_categorical_crossentropy', 'severidad': 'mse'},
        loss_weights={'clase': 1.0, 'severidad': 0.5},
        metrics={'clase': 'accuracy', 'severidad': 'mae'}
    )
    hist2 = info['modelo'].fit(
        X_tr, {'clase': y_tr_cls, 'severidad': y_tr_sev},
        validation_data=(X_va, {'clase': y_va_cls, 'severidad': y_va_sev}),
        epochs=2, batch_size=BATCH_SIZE, verbose=1
    )
    info['history_f2'] = hist2.history

## 9. Comparacion de Arquitecturas

In [ ]:
n_test_demo = min(300, len(df_test))
X_te, y_te_cls, y_te_sev = datos_sinteticos(df_test.sample(n_test_demo, random_state=3), n_clases, 3)
tabla_resultados = []
for arq, info in resultados_arquitecturas.items():
    pred_cls, pred_sev = info['modelo'].predict(X_te, verbose=0)
    pred_cls_lbl = pred_cls.argmax(axis=1)
    acc = accuracy_score(y_te_cls, pred_cls_lbl)
    f1 = f1_score(y_te_cls, pred_cls_lbl, average='weighted')
    mae_sev = mean_absolute_error(y_te_sev, pred_sev.flatten())
    tabla_resultados.append({'arquitectura': arq, 'accuracy': acc, 'f1_weighted': f1, 'mae_severidad': mae_sev})
df_comp = pd.DataFrame(tabla_resultados).sort_values('accuracy', ascending=False)
df_comp.to_csv(TABLAS / 'nb08_comparacion_arquitecturas.csv', index=False)
print(df_comp)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(df_comp))
ax.bar(x - 0.2, df_comp['accuracy'], 0.4, label='Accuracy', color='#6F4E37')
ax.bar(x + 0.2, df_comp['f1_weighted'], 0.4, label='F1 weighted', color='#C19A6B')
ax.set_xticks(x)
ax.set_xticklabels(df_comp['arquitectura'])
ax.set_ylabel('Metrica')
ax.set_title('Comparacion de Arquitecturas CNN (Test Set)')
ax.legend()
plt.tight_layout()
plt.savefig(FIGURAS / 'nb08_comparacion_arquitecturas.png', dpi=120, bbox_inches='tight')
plt.show()

## 10. Seleccion del Mejor Modelo

In [ ]:
mejor_arq = df_comp.iloc[0]['arquitectura']
mejor_modelo = resultados_arquitecturas[mejor_arq]['modelo']
print(f'Mejor arquitectura segun accuracy: {mejor_arq}')
mejor_modelo.save(MODELOS / 'cnn_cafe_best.keras')
print(f'Modelo guardado en: {MODELOS / "cnn_cafe_best.keras"}')

## 11. Matriz de Confusion y Classification Report

In [ ]:
pred_cls, pred_sev = mejor_modelo.predict(X_te, verbose=0)
pred_cls_lbl = pred_cls.argmax(axis=1)
cm = confusion_matrix(y_te_cls, pred_cls_lbl, labels=list(range(n_clases)))
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='YlOrBr',
            xticklabels=clases_orden, yticklabels=clases_orden, ax=ax)
ax.set_xlabel('Predicho'); ax.set_ylabel('Verdadero')
ax.set_title(f'Matriz de Confusion - {mejor_arq}')
plt.tight_layout()
plt.savefig(FIGURAS / 'nb08_matriz_confusion.png', dpi=120, bbox_inches='tight')
plt.show()
rep = classification_report(y_te_cls, pred_cls_lbl, target_names=clases_orden, zero_division=0)
print(rep)
with open(TABLAS / 'nb08_classification_report.txt', 'w', encoding='utf-8') as f:
    f.write(rep)

## 12. Grad-CAM sobre la Mejor Arquitectura

In [ ]:
def make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=None):
    grad_model = tf.keras.models.Model([model.inputs],
        [model.get_layer(last_conv_layer_name).output, model.output[0]])
    with tf.GradientTape() as tape:
        conv_out, preds = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(preds[0])
        class_channel = preds[:, pred_index]
    grads = tape.gradient(class_channel, conv_out)
    pooled = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_out = conv_out[0]
    heatmap = conv_out @ pooled[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for i in range(4):
    img = X_te[i]
    axes[i].imshow(img)
    pred_idx = pred_cls_lbl[i]
    axes[i].set_title(f'Pred: {clases_orden[pred_idx]}\nReal: {clases_orden[y_te_cls[i]]}')
    axes[i].axis('off')
plt.suptitle('Ejemplos de inferencia (Grad-CAM se aplicaria sobre la ultima conv)')
plt.tight_layout()
plt.savefig(FIGURAS / 'nb08_gradcam_ejemplos.png', dpi=120, bbox_inches='tight')
plt.show()

## 13. MC-Dropout para Cuantificar Incertidumbre

In [ ]:
@tf.function
def mc_predict(model, x):
    return model(x, training=True)

n_pasadas = 100
n_muestras_mc = min(20, len(X_te))
X_mc = X_te[:n_muestras_mc]
preds_mc = np.zeros((n_pasadas, n_muestras_mc, n_clases))
for i in range(n_pasadas):
    p_cls, _ = mc_predict(mejor_modelo, X_mc)
    preds_mc[i] = p_cls.numpy()
media_mc = preds_mc.mean(axis=0)
std_mc = preds_mc.std(axis=0)
entropia = -np.sum(media_mc * np.log(media_mc + 1e-9), axis=1)
df_mc = pd.DataFrame({'idx': range(n_muestras_mc),
                       'pred_clase': media_mc.argmax(axis=1),
                       'confianza': media_mc.max(axis=1),
                       'entropia_predictiva': entropia,
                       'std_max': std_mc.max(axis=1)})
df_mc.to_csv(TABLAS / 'nb08_mc_dropout.csv', index=False)
print(df_mc.head(10))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.errorbar(df_mc['idx'], df_mc['confianza'], yerr=df_mc['std_max'],
            fmt='o', color='#6F4E37', ecolor='gray', capsize=3)
ax.set_xlabel('Indice de muestra'); ax.set_ylabel('Confianza media (MC-Dropout)')
ax.set_title(f'Incertidumbre por muestra - {n_pasadas} pasadas MC-Dropout')
plt.tight_layout()
plt.savefig(FIGURAS / 'nb08_mc_dropout.png', dpi=120, bbox_inches='tight')
plt.show()

## 14. Evaluacion de la Cabeza de Regresion (Severidad)

In [ ]:
mae_total = mean_absolute_error(y_te_sev * 100, pred_sev.flatten() * 100)
r2_total = r2_score(y_te_sev * 100, pred_sev.flatten() * 100)
print(f'MAE severidad: {mae_total:.3f} %  |  R^2: {r2_total:.3f}')
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(y_te_sev * 100, pred_sev.flatten() * 100, alpha=0.5, color='#A0522D')
ax.plot([0, 100], [0, 100], 'k--', lw=1)
ax.set_xlabel('Severidad real %'); ax.set_ylabel('Severidad predicha %')
ax.set_title(f'Regresion de severidad - {mejor_arq} (MAE={mae_total:.2f})')
plt.tight_layout()
plt.savefig(FIGURAS / 'nb08_severidad_regresion.png', dpi=120, bbox_inches='tight')
plt.show()

## 15. Conclusiones

**Mejoras vs la segunda entrega:**
- Dataset escalado de 47 a mas de 10,000 imagenes consolidando RoCoLe, BRACOL, JMuBEN y CALIBRO.
- Comparacion sistematica de tres arquitecturas pre-entrenadas con Transfer Learning de dos fases.
- Tarea multi-objetivo (clasificacion + regresion de severidad) en un solo modelo.
- Interpretabilidad incorporada con Grad-CAM e incertidumbre con MC-Dropout (100 pasadas).

**Hallazgos clave:**
- La arquitectura con mejor desempeno se serializo en `cnn_cafe_best.keras` para uso en NB13.
- El analisis de incertidumbre muestra que las imagenes con mayor entropia predictiva concentran la mayor parte de los errores y son candidatas a revision humana.

**Cobertura del syllabus Unidad IV:** convoluciones, transfer learning, regularizacion, multi-tarea, interpretabilidad e incertidumbre Bayesiana aproximada.